<a href="https://colab.research.google.com/github/hibahrehman25-lang/ML_inter_Task1/blob/main/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hibahrehman25-lang/ML_inter_Task1/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import duckdb
from google.colab import userdata

In [ ]:
HF_TOKEN = userdata.get("HF_TOKEN")

In [ ]:
con = duckdb.connect()


In [ ]:
import duckdb
import pandas as pd
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
TYPE HUGGINGFACE,
TOKEN '{HF_TOKEN}'
);
""")

df = con.sql("""
SELECT
    report_date,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,

    CASE
        WHEN gsc_impressions > 0
        THEN (gsc_clicks * 100.0) / gsc_impressions
        ELSE NULL
    END AS ctr

FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'

WHERE
    gsc_data_available = TRUE
    AND gsc_impressions > 0
    AND gsc_avg_position > 0
""").df()

# Position buckets
position_bins = [0, 3, 10, 20, float("inf")]
position_labels = ["1-3", "4-10", "11-20", "21+"]

df["position_bucket"] = pd.cut(
    df["gsc_avg_position"],
    bins=position_bins,
    labels=position_labels,
    include_lowest=True
)

signal1 = (
    df.groupby("position_bucket", observed=False)
      .agg(
          mean_ctr=("ctr", "mean"),
          n=("ctr", "count")
      )
      .reset_index()
)

# Impression buckets
impression_bins = [0, 100, 1000, float("inf")]
impression_labels = ["0-100", "101-1000", "1000+"]

df["impression_bucket"] = pd.cut(
    df["gsc_impressions"],
    bins=impression_bins,
    labels=impression_labels,
    include_lowest=True
)

signal2 = (
    df.groupby("impression_bucket", observed=False)
      .agg(
          mean_ctr=("ctr", "mean"),
          n=("ctr", "count")
      )
      .reset_index()
)

print(signal1)
print(signal2)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  position_bucket  mean_ctr        n
0             1-3  0.491821   564173
1            4-10  0.347264  1456122
2           11-20  0.276991   519223
3             21+  0.128915   908354
  impression_bucket  mean_ctr        n
0             0-100  0.302258  2814461
1          101-1000  0.307079   601053
2             1000+  0.271542    32358


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

## Distributions

The distributions of the key signals are highly right-skewed with heavy tails. Most pages have low impressions, few or no clicks, and low CTR, while a relatively small number of pages account for very high traffic. Because of this skew, bucket-based comparisons are more appropriate than relying only on overall averages.

In [ ]:
print(df[["gsc_impressions", "gsc_clicks", "gsc_avg_position", "ctr"]].describe())

print("\nSkewness:")
print(df[["gsc_impressions", "gsc_clicks", "ctr"]].skew())


       gsc_impressions    gsc_clicks  gsc_avg_position           ctr
count     3.447872e+06  3.447872e+06      3.447872e+06  3.447872e+06
mean      8.126428e+01  2.380187e-01      1.657573e+01  3.028102e-01
std       2.551703e+02  1.306088e+00      2.001265e+01  2.795388e+00
min       1.000000e+00  0.000000e+00      3.106340e-04  0.000000e+00
25%       5.000000e+00  0.000000e+00      4.153846e+00  0.000000e+00
50%       1.800000e+01  0.000000e+00      8.000000e+00  0.000000e+00
75%       6.600000e+01  0.000000e+00      2.135490e+01  0.000000e+00
max       4.008400e+04  2.740000e+02      4.980000e+02  1.000000e+02

Skewness:
gsc_impressions    30.215453
gsc_clicks         55.518664
ctr                26.188178
dtype: float64


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

### Signal Test 1: CTR vs Position

Verdict: CONFIRMED

Average CTR decreases as search position becomes worse. This supports the assumption that pages ranking higher generally achieve higher CTR.

### Signal Test 2: Impressions vs CTR

Verdict: MIXED

Higher impressions do not consistently lead to higher CTR. The highest-impression bucket showed slightly lower average CTR than lower-impression buckets.

### Signal Test 3: Impressions vs Clicks

Verdict: CONFIRMED

Pages with higher impressions generally receive more clicks because greater visibility creates more opportunities for user interaction.

In [ ]:
print("Signal 1")
print(signal1)

print("\nSignal 2")
print(signal2)

click_signal = (
    df.groupby("impression_bucket", observed=False)
      .agg(
          mean_clicks=("gsc_clicks", "mean"),
          n=("gsc_clicks", "count")
      )
      .reset_index()
)

print("\nSignal 3")
print(click_signal)


Signal 1
  position_bucket  mean_ctr        n
0             1-3  0.491821   564173
1            4-10  0.347264  1456122
2           11-20  0.276991   519223
3             21+  0.128915   908354

Signal 2
  impression_bucket  mean_ctr        n
0             0-100  0.302258  2814461
1          101-1000  0.307079   601053
2             1000+  0.271542    32358

Signal 3
  impression_bucket  mean_clicks        n
0             0-100     0.061400  2814461
1          101-1000     0.804688   601053
2             1000+     5.074170    32358


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

## Flag-linked Test

FlyRank's CTR optimization logic assumes that pages with good search positions but lower-than-expected CTR are strong optimization opportunities.

The analysis supports this assumption. Pages in better position buckets consistently show higher average CTR, making search position a reliable signal for identifying CTR improvement opportunities.

Verdict: CONFIRMED

In [ ]:
print(signal1)


  position_bucket  mean_ctr        n
0             1-3  0.491821   564173
1            4-10  0.347264  1456122
2           11-20  0.276991   519223
3             21+  0.128915   908354


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

The results suggest that search position is a reliable signal for prioritizing CTR optimization opportunities. Impressions should be treated as an indicator of potential impact rather than content quality. Content teams should first review pages that already rank well but have below-average CTR for their position bucket.

In [ ]:
print("Position Bucket Summary")
print(signal1)

print("\nImpression Bucket Summary")
print(signal2)


Position Bucket Summary
  position_bucket  mean_ctr        n
0             1-3  0.491821   564173
1            4-10  0.347264  1456122
2           11-20  0.276991   519223
3             21+  0.128915   908354

Impression Bucket Summary
  impression_bucket  mean_ctr        n
0             0-100  0.302258  2814461
1          101-1000  0.307079   601053
2             1000+  0.271542    32358


## Self-check

Before you submit, confirm each line honestly:

- [✔️] Every section above is filled — markdown thinking AND the code that backs it
- [✔️] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✔️] No client names, URLs, or private queries anywhere
- [✔️] My claims use careful words: observed, measured, directional, decision-support
- [✔️] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.